In [ ]:
# CELL 1: Imports e configuração
import logging
import pandas as pd

from functions.config import ProjectConfig
from functions.Diretories_Downloads import dir_project, cif_download
from functions.Primary_Filter import primary_filter
from functions.Extraction_Phases import extrair_fracoes_fase

# Configura logging para exibir mensagens de INFO e acima no notebook
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)

In [ ]:
# CELL 2: Cria o diretório do projeto

cfg = ProjectConfig("projeto_x")
cfg.create_dirs()

print(cfg.project_dir)
print(cfg.ref_dir)
print(cfg.results_dir)

In [ ]:
# CELL 3: Download referências

dict_refs = {
    # alpha-Fe2O3: Trigonal, R-3c. Referência clássica de alta cristalinidade.
    "Hematite": 2101168,

    # Fe3O4: Cúbico, Fd-3m. Estrutura de espinélio invertido.
    # Cuidado: Picos muito próximos aos da Maghemita.
    "Magnetite": 9005813,

    # gamma-Fe2O3: Cúbico, P4332. Fase metaestável.
    # Importante conferir se não há picos de superestrutura no seu DRX.
    "Maghemite": 9017489,

    # alpha-FeOOH: Ortorrômbico, Pnma. Comum se houver hidratação no resíduo.
    "Goethite": 7720752,

    # FeO: Cúbico, Fm-3m. FASE CRÍTICA para a redução a 850 °C.
    "Wüstite": 9002673,

    # Fe3C: Ortorrômbico, Pnma. Fase crítica para a redução a 850 °C.
    "Cementite": 9016584,

    # alpha-Fe: Cúbico, Im-3m. O produto final da redução (Ferro metálico).
    "Fe_metalico": 9008536,
}

for Nome, cod_ID in dict_refs.items():
    cif_download(Nome, cod_ID, str(cfg.ref_dir))

In [ ]:
# CELL 4: Filtragem primária

input_drx_raw = "inputs/hematite_raw.txt"
df, amostra_norm, theta = primary_filter(
    correlation="pearson",
    input_file=input_drx_raw,
    ref_dir=str(cfg.ref_dir),
)
print(df)

In [ ]:
# CELL 5: Plot

for i, row in df.iterrows():
    print(f"Referência: {row['nome']}, Score: {row['score']:.2%}")
    score = row['score']
    nome = row['nome']
    ref_int_norm = row['ref_int_norm']

    # Plot
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(theta, amostra_norm, label="Amostra (experimental)", linewidth=0.8)
    ax.plot(theta, ref_int_norm, label=f"Referência: {nome} (score={score:.2%})", linewidth=0.8, alpha=0.8)
    ax.set_xlabel("2θ (°)")
    ax.set_ylabel("Intensidade Normalizada")
    ax.set_title("XRD: Amostra vs Melhor Referência")
    ax.legend()
    plt.tight_layout()
    plt.show()   

    break

In [ ]:
# CELL 6: Refinamento Rietveld (GSAS-II)

import importlib
import functions.Workflow_GSAS2 as wf
importlib.reload(wf)
from functions.Workflow_GSAS2 import refinamento_sequencial_oxidos

refs_cif = [
    str(cfg.ref_dir / df['nome'][0]),
    str(cfg.ref_dir / df['nome'][1]),
    # str(cfg.ref_dir / df['nome'][2]),
    # str(cfg.ref_dir / df['nome'][3]),
]

input_inst = "inputs/inst_rruff.prm"
resultado = refinamento_sequencial_oxidos(
    input_drx_raw,
    input_inst,
    refs_cif,
    cfg.project_name,
    refinar_atomos=True,  # mude para False se o XRD for de baixa qualidade
)

In [ ]:
# CELL 7: Plot do Refinamento de Rietveld

import matplotlib.pyplot as plt

x = resultado["x"]
yobs = resultado["yobs"]
ycalc = resultado["ycalc"]
diff = resultado["diff"]
wR = resultado["wR"]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), height_ratios=[3, 1],
                                sharex=True, gridspec_kw={"hspace": 0.05})

# Painel superior: observado vs calculado
ax1.plot(x, yobs, "k-", label="Observado", linewidth=0.6)
ax1.plot(x, ycalc, "r-", label="Calculado", linewidth=0.6)
ax1.set_ylabel("Intensidade")
ax1.set_title(f"Refinamento de Rietveld — wR = {wR:.2f}%")
ax1.legend()

# Painel inferior: diferença
ax2.plot(x, diff, "b-", linewidth=0.5)
ax2.axhline(0, color="gray", linewidth=0.5, linestyle="--")
ax2.set_xlabel("2θ (°)")
ax2.set_ylabel("Diferença")

plt.tight_layout()
plt.show()

In [ ]:
# Função para plotar o refinamento calculado junto com as curvas referenciais usadas
from functions.Primary_Filter import simular_padrao_xrd

def plot_refinamento_com_referencias(x, yobs, ycalc, refs_cif, theta, ref_dir):
    import matplotlib.pyplot as plt
    # Normalização das curvas do refinamento
    yobs_norm = (yobs - yobs.min()) / (yobs.max() or 1.0)
    ycalc_norm = (ycalc - ycalc.min()) / (ycalc.max() or 1.0)
    plt.figure(figsize=(12, 6))
    plt.plot(x, yobs_norm, 'k-', label='Observado', linewidth=0.8)
    plt.plot(x, ycalc_norm, 'r-', label='Calculado (Rietveld)', linewidth=0.8)
    
    # Plotar as curvas referenciais simuladas
    for ref_cif in refs_cif:
        nome_fase = ref_cif.split('/')[-1].replace('.cif', '')
        try:
            yref = simular_padrao_xrd(ref_cif, theta)
            yref_norm = (yref - yref.min()) / (yref.max() or 1.0)
            plt.plot(theta, yref_norm, '--', label=f'Referência: {nome_fase}', alpha=0.7)
        except Exception as e:
            print(f"Erro ao simular referência {nome_fase}: {e}")
    
    plt.xlabel('2θ (°)')
    plt.ylabel('Intensidade Normalizada')
    plt.title('Refinamento de Rietveld e Referências')
    plt.legend()
    plt.tight_layout()
    plt.show()

# Exemplo de uso:
plot_refinamento_com_referencias(x, yobs, ycalc, refs_cif, theta, dir_ref)


In [ ]:
# CELL 8: Extração automática dos percentuais de fase do .lst

lst_file_path = str(cfg.results_dir / f"{cfg.project_name}.lst")

dados_extraidos = extrair_fracoes_fase(lst_file_path)

print("--- Resultados do Refinamento ---")
if isinstance(dados_extraidos, dict):
    for fase, fracoes in dados_extraidos.items():
        print(f"\nFase: {fase}")
        print(f"  Fração da Fase (Phase Fraction): {fracoes['Phase Fraction']}")

        weight_pct = fracoes['Weight Fraction'] * 100
        print(f"  Fração em Peso (Weight Fraction): {fracoes['Weight Fraction']} ({weight_pct:.2f}%)")
else:
    print(dados_extraidos)